In [ ]:
# Function "scatter_2d()"
def scatter_2d(
    df,
    x,
    y,
    color=None,
    size=None,
    alpha=None,
    shape=None,
    markers_list=[
        "o",
        "$X$",
        "*",
        "^",
        "s",
        "p",
        "1",
        "8",
    ],
    fitline=None,
    edgecolor="none",
):
    """
    Create a 2D scatter plot with customizable aesthetics (color, size, transparency, shape) and dynamic legends.

    Parameters:
    -----------
    df : pandas.DataFrame
        Input DataFrame containing the data to plot.

    x : str
        Name of the column in `df` to use as the x-axis variable.

    y : str
        Name of the column in `df` to use as the y-axis variable.

    color : str, int, float, or None, optional (default=None)
        - If `str`: Name of a column in `df` to map to point colors. A legend is generated.
        - If `int/float`: Fixed color value (e.g., "blue") applied to all points. No legend.
        - If `None`: Default color is "blue".

    size : str, int, float, or None, optional (default=None)
        - If `str`: Name of a column in `df` to map to point sizes. A legend is generated.
        - If `int/float`: Fixed size value applied to all points. No legend.
        - If `None`: Default size is 30.

    alpha : str, int, float, or None, optional (default=None)
        - If `str`: Name of a column in `df` to map to point transparency (alpha). A legend is generated.
        - If `int/float`: Fixed alpha value (0.0 to 1.0) applied to all points. No legend.
        - If `None`: Default alpha is 0.9.

    shape : str, or None, optional (default=None)
        - If `str`: Name of a column in `df` to map to point shapes. A legend is generated.
        - If `None`: Default shape is "o" (circle).

    markers_list : list of str, optional (default=predefined list)
        List of matplotlib-compatible marker symbols to use for categorical shape variables.
        Only used if `shape_var` is a column name.

    fitline : str or None, optional (default=None)
        If `linear`, adds a linear regression fit line (red solid line) to the plot.
        If `quadratic`, adds an exponential fit line (purple dotted) to the plot.
        If `both`, adds both.
        If None, adds nothing.

    edgecolor: str, int, float, or None, optional (default="none")
        - Controls the edge color of the pointers

    Behavior:
    ---------
    - Legends are dynamically generated for variables that are mapped to categorical/column-based values.
    - Fixed values (e.g., "blue", 100, 0.5) do not generate legends.
    - Legends are stacked vertically on the right side of the plot with calculated spacing.
    - Optional regression fit line can be added for data summarization.

    Returns:
    --------
    - None

    Figures:
    --------
    - Scatterplot.
    """

    # ======================================================================================
    # Checking input parameters
    # ======================================================================================

    # Checking if "df" is a Pandas Dataframe
    if not isinstance(df, pd.DataFrame):
        raise TypeError(
            f'The variable "df" should be a Pandas DataFrame, not a "{type(df).__name__}".'
        )

    # Random seed
    np.random.seed(42)

    # Define x, and y variables
    x = df[x]
    y = df[y]

    # ======================================================================================
    # Define additional variables to control other elements of the plot
    # ======================================================================================

    # Alpha
    need_alpha_legend = 0
    if type(alpha) == str:
        alpha_var = df[alpha]
        alpha_array = normalize(alpha_var, 0.1)
        need_alpha_legend = 1
    elif isinstance(alpha, (int, float)):
        alpha_array = pd.Series([alpha] * len(df))
    elif alpha is None:
        alpha_array = pd.Series([0.3] * len(df))

    # Size
    need_size_legend = 0
    if type(size) == str:
        size_var = df[size]
        size_array = normalize(size_var, 4, 1000)
        need_size_legend = 1
    elif isinstance(size, (int, float)):
        size_array = pd.Series([size] * len(df))
    elif size is None:
        size_array = pd.Series([30] * len(df))

    # Color
    need_color_legend = 0
    if color in df.columns:
        color_var = df[color]
        unique_color_categories = np.unique(color_var)
        hex_colors = generate_hex_colors(len(unique_color_categories))
        custom_colormap = mpl.colors.ListedColormap(hex_colors)
        color_numbers = np.random.rand(len(unique_color_categories))
        color_mapping = {
            color_category: number
            for color_category, number in zip(
                unique_color_categories,
                color_numbers,
            )
        }
        color_array = color_var.map(color_mapping)
        need_color_legend = 1
    elif type(color) == str:
        color_array = pd.Series([color] * len(df))
    elif color is None:
        color_array = pd.Series(["blue"] * len(df))

    # Shape
    need_shape_legend = 0
    if shape in df.columns:
        shape_var = df[shape]
        unique_shape_categories = np.unique(shape_var)
        selected_markers = markers_list[: len(unique_shape_categories)]
        shape_mapping = {
            shape_category: marker
            for shape_category, marker in zip(
                unique_shape_categories,
                selected_markers,
            )
        }
        shape_array = shape_var.map(shape_mapping)
        need_shape_legend = 1
    elif type(shape) == str:
        pass
    elif shape is None:
        shape_var = "o"

    # ======================================================================================
    # Plot
    # ======================================================================================

    fig, ax = mpl.pyplot.subplots(figsize=(14, 10))

    if need_shape_legend:
        for (
            shape_category,
            marker,
        ) in shape_mapping.items():
            mask = shape_var == shape_category
            ax.scatter(
                x[mask],
                y[mask],
                c=color_array[mask],
                cmap=custom_colormap if need_color_legend else None,
                s=size_array[mask],
                alpha=alpha_array[mask],
                edgecolor=edgecolor,
                marker=marker,
                label=shape_category,
                linewidth=1.5,
            )
    else:
        ax.scatter(
            x,
            y,
            c=color_array,
            cmap=custom_colormap if need_color_legend else None,
            s=size_array,
            alpha=alpha_array,
            edgecolor=edgecolor,
            marker=shape_var,
            linewidth=1.5,
        )

    # ======================================================================================
    # Creating legends for the 4 additional variables
    # ======================================================================================

    # Color legend
    if need_color_legend:
        color_legend_elements = [
            mpl.patches.Patch(
                facecolor=hex_colors[i],
                edgecolor="black",
                label=cat,
            )
            for i, cat in enumerate(unique_color_categories)
        ]

    # Alpha legend
    if need_alpha_legend:
        alpha_min = alpha_array.min()
        alpha_max = alpha_array.max()
        alpha_mid = (alpha_max + alpha_min) / 2
        alpha_legend_elements = [
            mpl.lines.Line2D(
                [0],
                [0],
                marker="o",
                color="blue"
                if need_color_legend
                else color_array.iloc[1],
                label="Low",
                markerfacecolor="blue"
                if need_color_legend
                else color_array.iloc[1],
                markersize=10,
                alpha=alpha_min,
                linestyle="",
            ),
            mpl.lines.Line2D(
                [0],
                [0],
                marker="o",
                color="blue"
                if need_color_legend
                else color_array.iloc[1],
                label="Mid",
                markerfacecolor="blue"
                if need_color_legend
                else color_array.iloc[1],
                markersize=10,
                alpha=alpha_mid,
                linestyle="",
            ),
            mpl.lines.Line2D(
                [0],
                [0],
                marker="o",
                color="blue"
                if need_color_legend
                else color_array.iloc[1],
                label="High",
                markerfacecolor="blue"
                if need_color_legend
                else color_array.iloc[1],
                markersize=10,
                alpha=alpha_max,
                linestyle="",
            ),
        ]

    # Size legend
    if need_size_legend:
        size_min = size_array.quantile(0.01)
        size_max = size_array.quantile(0.75)
        size_mid = (size_min + size_max) / 2
        size_legend_elements = [
            mpl.lines.Line2D(
                [0],
                [0],
                marker="o",
                color="blue"
                if need_color_legend
                else color_array.iloc[1],
                label="Small",
                markerfacecolor="blue"
                if need_color_legend
                else color_array.iloc[1],
                markersize=np.sqrt(size_min),
                linestyle="",
            ),
            mpl.lines.Line2D(
                [0],
                [0],
                marker="o",
                color="blue"
                if need_color_legend
                else color_array.iloc[1],
                label="Medium",
                markerfacecolor="blue"
                if need_color_legend
                else color_array.iloc[1],
                markersize=np.sqrt(size_mid),
                linestyle="",
            ),
            mpl.lines.Line2D(
                [0],
                [0],
                marker="o",
                color="blue"
                if need_color_legend
                else color_array.iloc[1],
                label="Big",
                markerfacecolor="blue"
                if need_color_legend
                else color_array.iloc[1],
                markersize=np.sqrt(size_max),
                linestyle="",
            ),
        ]

    # Shape legend
    if need_shape_legend:
        shape_legend_elements = [
            mpl.lines.Line2D(
                [0],
                [0],
                marker=marker,
                label=label,
                color="blue"
                if need_color_legend
                else color_array.iloc[1],
                markerfacecolor="blue"
                if need_color_legend
                else color_array.iloc[1],
                markersize=10,
                linestyle="",
            )
            for label, marker in shape_mapping.items()
        ]

    # ======================================================================================
    # Inserting legends keeping spacing equal
    # ======================================================================================

    # Define starting point
    x_legend = 1
    y_legend = 1
    loc = "upper left"
    prop = {"size": 12}
    gap = 0.015

    # Add color legend to the plot
    if need_color_legend:
        color_legend = fig.legend(
            handles=color_legend_elements,
            title=color_var.name,
            bbox_to_anchor=(x_legend, y_legend),
            loc=loc,
            prop=prop,
            bbox_transform=ax.transAxes,
        )

        # Calculate next y coordinate
        y_legend = calculate_legend_next_y_anchor(
            figure=fig,
            axes=ax,
            legend=color_legend,
            y_top=y_legend,
            gap=gap,
        )

    # Add alpha legend to the plot
    if need_alpha_legend:
        alpha_legend = fig.legend(
            handles=alpha_legend_elements,
            title=alpha_var.name,
            bbox_to_anchor=(x_legend, y_legend),
            loc=loc,
            prop=prop,
            bbox_transform=ax.transAxes,
        )
        ax.add_artist(alpha_legend)

        # Calculate next y coordinate
        y_legend = calculate_legend_next_y_anchor(
            figure=fig,
            axes=ax,
            legend=alpha_legend,
            y_top=y_legend,
            gap=gap,
        )

    # Add size legend to the plot
    if need_size_legend:
        size_legend = fig.legend(
            handles=size_legend_elements,
            title=size_var.name,
            bbox_to_anchor=(x_legend, y_legend),
            loc=loc,
            prop=prop,
            labelspacing=2,
            bbox_transform=ax.transAxes,
        )
        ax.add_artist(size_legend)

        # Calculate next y coordinate
        y_legend = calculate_legend_next_y_anchor(
            figure=fig,
            axes=ax,
            legend=size_legend,
            y_top=y_legend,
            gap=gap,
        )

    # Add shape legend to the plot
    if need_shape_legend:
        shape_legend = fig.legend(
            handles=shape_legend_elements,
            title=shape_var.name,
            bbox_to_anchor=(x_legend, y_legend),
            loc=loc,
            prop=prop,
            bbox_transform=ax.transAxes,
        )
        ax.add_artist(shape_legend)

    # ======================================================================================
    # Add fit line
    # ======================================================================================

    # Linear fitline
    if fitline == "linear":
        slope, intercept, _, _, _ = sp.stats.linregress(
            x.values, y.values
        )
        x_fit = np.linspace(x.values.min(), x.values.max(), 100)
        y_fit = intercept + slope * x_fit
        ax.plot(
            x_fit,
            y_fit,
            color="red",
            linestyle="-",
            linewidth=2.5,
            label="Linear Fitline",
        )

        fitline_legend_elements = [
            mpl.lines.Line2D(
                [0],
                [0],
                color="red",
                linestyle="-",
                linewidth=2.5,
                label="Linear Fitline",
            )
        ]

        fitline_legend = fig.legend(
            handles=fitline_legend_elements,
            bbox_to_anchor=(0, 0),
            loc="lower right",
            bbox_transform=ax.transAxes,
        )
        ax.add_artist(fitline_legend)

    # Quadratic fitline
    elif fitline == "quadratic":

        def quadratic_func(x, a, b, c):
            return a * x**2 + b * x + c

        popt, _ = sp.optimize.curve_fit(
            quadratic_func, x.values, y.values
        )
        x_fit = np.linspace(x.values.min(), x.values.max(), 100)
        y_fit = quadratic_func(x_fit, *popt)
        ax.plot(
            x_fit,
            y_fit,
            color="purple",
            linestyle="--",
            linewidth=2,
            label="Quadratic Fitline",
        )

        fitline_legend_elements = [
            mpl.lines.Line2D(
                [0],
                [0],
                color="purple",
                linestyle="--",
                linewidth=2.5,
                label="Quadratic Fitline",
            )
        ]

        fitline_legend = fig.legend(
            handles=fitline_legend_elements,
            bbox_to_anchor=(0, 0),
            loc="lower right",
            bbox_transform=ax.transAxes,
        )
        ax.add_artist(fitline_legend)

    # Both fitlines
    elif fitline == "both":
        # Linear Fitline
        slope, intercept, _, _, _ = sp.stats.linregress(
            x.values, y.values
        )
        x_fit = np.linspace(x.values.min(), x.values.max(), 100)
        y_fit = intercept + slope * x_fit
        ax.plot(
            x_fit,
            y_fit,
            color="red",
            linestyle="-",
            linewidth=2.5,
            label="Linear Fitline",
        )

        lin_fitline_legend_elements = [
            mpl.lines.Line2D(
                [0],
                [0],
                color="red",
                linestyle="-",
                linewidth=2.5,
                label="Linear Fitline",
            )
        ]

        lin_fitline_legend = fig.legend(
            handles=lin_fitline_legend_elements,
            bbox_to_anchor=(0, 0),
            loc="lower right",
            bbox_transform=ax.transAxes,
        )
        ax.add_artist(lin_fitline_legend)

        # Quadratic fitline
        def quadratic_func(x, a, b, c):
            return a * x**2 + b * x + c

        popt, _ = sp.optimize.curve_fit(
            quadratic_func, x.values, y.values
        )
        x_fit = np.linspace(x.values.min(), x.values.max(), 100)
        y_fit = quadratic_func(x_fit, *popt)
        ax.plot(
            x_fit,
            y_fit,
            color="purple",
            linestyle="--",
            linewidth=2,
            label="Quadratic Fitline",
        )

        quad_fitline_legend_elements = [
            mpl.lines.Line2D(
                [0],
                [0],
                color="purple",
                linestyle="--",
                linewidth=2.5,
                label="Quadratic Fitline",
            )
        ]

        quad_fitline_legend = fig.legend(
            handles=quad_fitline_legend_elements,
            bbox_to_anchor=(0, 0.05),
            loc="lower right",
            bbox_transform=ax.transAxes,
        )
        ax.add_artist(quad_fitline_legend)

    # ======================================================================================
    # Final touches: Add title and axis labels
    # ======================================================================================

    ax.set_title(
        f'2D Scatter Plot with Custom Aesthetics:\n"{y.name}" ~ "{x.name}"',
        fontsize=14,
        pad=20,
    )
    ax.set_xlabel(f'X - Variable: "{x.name}"', fontsize=12)
    ax.set_ylabel(f'Y - Variable: "{y.name}"', fontsize=12)

'uehgueihwuhvuehvhfvkshjvhdfkjvhejhgvejhgvkefhksjhksjdhjfhsjhgkfjjjkkkkkkkkkkkkkkkkkkk'

In [ ]:
def generate_hex_colors(n=40):
    np.random.seed(42)
    hex_colors = []
    for i in range(n):
        r, g, b = np.random.randint(0, 256, size=3)
        hex_color = f"#{r:02X}{g:02X}{b:02X}"
        hex_colors.append(hex_color)

    return hex_colors


def calculate_legend_next_y_anchor(figure, axes, legend, y_top, gap=0.02):

    # Force draw figure
    figure.canvas.draw()

    # Create renderer
    renderer = figure.canvas.get_renderer()

    # Find dislpay coordinatef of the legend bounding box
    bbox_display_coord = legend.get_window_extent(renderer=renderer)

    # Transform the display coordinates in axes coordinates
    # Invert the axes
    inv_axes = axes.transAxes.inverted()

    # Transform coordinates
    _, y_bottom_axes_coord = inv_axes.transform((0.0, bbox_display_coord.y0))
    _, y_top_axes_coord = inv_axes.transform((0.0, bbox_display_coord.y1))

    # Calculate bbox height in axes coordinates
    bbox_height = y_top_axes_coord - y_bottom_axes_coord

    # Calculate the next y_top
    next_y_top = y_top - bbox_height - gap

    # Return
    return next_y_top
